<a href="https://colab.research.google.com/github/Faizybro/Faizybro/blob/main/feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load and Preprocess the Dataset

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("vehicle dataset.csv")

# Encode categorical variables
label_encoders = {}
for col in ['Gender', 'Vehicle_Age', 'Vehicle_Damage']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Separate features and target
X = df.drop(['id', 'Response'], axis=1)
y = df['Response']

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


Feature Extraction using PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Apply PCA
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Train a classifier on PCA-transformed data
clf_pca = LogisticRegression(max_iter=1000)
clf_pca.fit(X_train_pca, y_train)
y_pred_pca = clf_pca.predict(X_test_pca)

# Evaluate accuracy
accuracy_pca = accuracy_score(y_test, y_pred_pca)
print(f"Accuracy using PCA: {accuracy_pca:.4f}")


Accuracy using PCA: 0.8751


Feature Extraction using Autoencoder

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Define the Autoencoder model
input_dim = X_train.shape[1]
encoding_dim = 5

input_layer = Input(shape=(input_dim,))
encoded = Dense(8, activation='relu')(input_layer)
encoded = Dense(encoding_dim, activation='relu')(encoded)
decoded = Dense(8, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
encoder = Model(inputs=input_layer, outputs=encoded)

autoencoder.compile(optimizer='adam', loss='mse')

# Train the autoencoder
autoencoder.fit(X_train, X_train, epochs=10, batch_size=256, shuffle=True, validation_split=0.2, verbose=1)


Epoch 1/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.7628 - val_loss: 0.3030
Epoch 2/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2860 - val_loss: 0.2594
Epoch 3/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.2544 - val_loss: 0.2405
Epoch 4/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.2344 - val_loss: 0.2197
Epoch 5/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.2159 - val_loss: 0.2117
Epoch 6/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.2106 - val_loss: 0.2063
Epoch 7/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.2036 - val_loss: 0.1980
Epoch 8/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1950 - val_loss: 0.1831
Epoch 9/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1729 - val_loss: 0.1328
Epoch 10/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1326 - val_loss: 0.1269


Classification report

In [ ]:
from sklearn.metrics import classification_report


In [ ]:
print("=== Classification Report: PCA ===")
print(classification_report(y_test, y_pred_pca, target_names=["No Response", "Response"]))


=== Classification Report: PCA ===
              precision    recall  f1-score   support

 No Response       0.88      1.00      0.93     66699
    Response       0.00      0.00      0.00      9523

    accuracy                           0.88     76222
   macro avg       0.44      0.50      0.47     76222
weighted avg       0.77      0.88      0.82     76222



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("=== Classification Report: Autoencoder ===")
print(classification_report(y_test, y_pred_ae, target_names=["No Response", "Response"]))


=== Classification Report: Autoencoder ===
              precision    recall  f1-score   support

 No Response       0.88      1.00      0.93     66699
    Response       0.48      0.00      0.00      9523

    accuracy                           0.88     76222
   macro avg       0.68      0.50      0.47     76222
weighted avg       0.83      0.88      0.82     76222

